# A 1Grp Response

In this example we will explore leveraging an adjoint formulation of the linear boltzmann equations for computing response functions in a desired region of interest (RoI).

This is a complete simulation transport example. Each aspect of this example can be broken as follows:
- [Prerequisites](#prerequisites)

- [Geometry](#Geometry)
    - [Mesh](#mesh)
    - [Material IDs](#material-ids)
    - [Volumetric Source](#volumetric-source)
- [Solver](#Solver)
    - [Angular Quadrature](#angular-quadrature)
    - [Linear Boltzmann Solver](#linear-boltzmann-solver)
    - [A Region of Interest](#a-region-of-interest)
- [Adjoint Solver](#adjoint-solver)
    - [Adjoint Source](#adjoint-source)
    - [Adjoint Options](#adjoint-options)
    - [Evaluate Resposne](#evaluate-response)
- [Solution](#solution) 

## Prerequisites

Before running this example, make sure that the **Python module of OpenSn** was installed.

### Converting and Running this Notebook from the Terminal
To run this notebook from the terminal, simply type:

`jupyter nbconvert --to python --execute VolSrc_1D.ipynb`.

To run this notebook in parallel (for example, using 4 processes), simply type:

`mpiexec -n 4 jupyter nbconvert --to python --execute VolSrc_1D.ipynb`.

In [ ]:
from mpi4py import MPI
size = MPI.COMM_WORLD.size
rank = MPI.COMM_WORLD.rank
barrier = MPI.COMM_WORLD.barrier

if rank == 0:
    print(f"Running the first LBS Adjoint example with {size} MPI processors.")

Running the first LBS Adjoint example with 1 MPI processors.


### Import Requirements

Import required classes and functions from the Python interface of OpenSn. Make sure that the path
to PyOpenSn is appended to Python's PATH.

In [ ]:
# assuming that the execute dir is the notebook dir
# this line is not necessary when PyOpenSn is installed using pip
sys.path.append("../../../..")

from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.xs import MultiGroupXS
from pyopensn.source import PointSource, VolumetricSource
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSolver
from pyopensn.response import ResponseEvaluator
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.context import UseColor, Finalize


import os
import sys
import numpy as np

## Geometry
### Mesh
Here, we will use the in-house orthogonal mesh generator for a simple Cartesian grid. We first create a list of nodes in the X direction.

In [ ]:
N = 100
L = 10.0
ds = L / N
nodes = []
for i in range(N + 1):
    nodes.append(i * ds)

### Orthogonal Mesh Generation
We use the `OrthogonalMeshGenerator` and pass the list of nodes per dimension. Here, we pass our lsit of nodes for the x direction creating a 1D geometry of side length 10 with 100 square cells. 

In [ ]:
meshgen = OrthogonalMeshGenerator( node_sets=[nodes] )
grid = meshgen.Execute()

### Material IDs
When using the in-house `OrthogonalMeshGenerator`, no material IDs are assigned. The user needs to assign material IDs to all cells. We will begin by assigning the background material ID with a value of 0.

In [ ]:
background = RPPLogicalVolume(infx=True, infy=True, infz=True)
grid.SetBlockIDFromLogicalVolume(background, 0, True)

Next we will assign material IDs for the source and detector. When assigning material IDs via logical volumes the IDs for each cell is overriden by the most recent assignment. That is to say, assigning a material ID of 1 to any region in our domain will override the previously set ID to those cells. 

In [ ]:
source = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=2.0, zmax=4.0,
)
grid.SetBlockIDFromLogicalVolume(source, 1, True)

det = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=7.75, zmax=8.25,
)
grid.SetBlockIDFromLogicalVolume(det, 2, True)

The resulting mesh and partition is shown below:
![Mesh_Partition](images/first_example_mesh_partition.png)

### Cross Sections
We create one-group cross sections using a built-in method. 
See the tutorials' section on cross sections for more details on how to load cross sections into OpenSn.

For this tutorial we will be using simple one-group cross sections. For the background we assigning vacuum. For the source and RoI, $\sigma_t$ is 1.0 where both the source and the RoI are pure absorbers. 

In [ ]:
xs_bkgrnd = MultiGroupXS()

# See if you can assign a nonzero c for sigma_t=0!!
xs_bkgrnd.CreateSimpleOneGroup(sigma_t=0.0, c=0.0)

xs_src = MultiGroupXS()
xs_src.CreateSimpleOneGroup(sigma_t=1.0, c=0.0)

xs_det = MultiGroupXS()
xs_det.CreateSimpleOneGroup(sigma_t=1.0, c=0.0)

### Volumetric Source
We create a volumetric one-group source be assigned to the associated block ID=1.

Volumetric sources are assigned to the solver via the `options` parameter in the LBS block.

In [ ]:
vol_src = VolumetricSource(block_ids=[1], group_strength=[1.0])

## Solver
### Angular Quadrature
Since we are solving a 1D problem we will create Gauss-Legendre Product Quadrature for a 1D slab which requires the total angles. In this case we will use 1024 angles creating a 1D angular quadrature in $\mu$.

In [ ]:
pquad = GLProductQuadrature1DSlab(1024)

### Linear Boltzmann Solver
#### Options for the Linear Boltzmann Problem (LBS)
In the LBS block, we provide
+ The number of energy groups
+ The groupsets (with 0-indexing);
    + the handle for the angular quadrature
    + the angle aggregation
    + the solver type
    + tolerances
+ Cross section map
+ Solver options

In [ ]:
phys = DiscreteOrdinatesProblem(
    mesh=grid,
    num_groups=1,
    groupsets=[
        {
            "groups_from_to": (0, 0),
            "angular_quadrature": pquad,
            "angle_aggregation_num_subsets": 1,
            "inner_linear_method": "petsc_gmres",
            "l_abs_tol": 1.0e-6,
            "l_max_its": 300,
            "gmres_restart_interval": 30
        }
    ],
    xs_map=[
        {"block_ids": [0], "xs": xs_bkgrnd},
        {"block_ids": [1], "xs": xs_src},
        {"block_ids": [2], "xs": xs_det}
    ],
    options={
        "scattering_order": 0,
        "spatial_discretization": "pwld",
        "boundary_conditions": [
            {"name": "zmin", "type": "vacuum"},
            {"name": "zmax", "type": "vacuum"}
        ],
        "volumetric_sources": [vol_src],
    },
)

### Putting the Linear Boltzmann Solver Together
We then create the physics solver, initialize it, and execute it.

In [ ]:
ss_solver = SteadyStateSolver(lbs_problem=phys)
ss_solver.Initialize()

Note: In this tutorial we have decided to execute the linear Boltzmann solver in addition to the adjoint solver to illustrate their duality. This is not necessary to execute the adjoint solver. 

In [ ]:
ss_solver.Execute()

### A Region of Interest

With the LBS solver executed, we now have solutions to the transport problem. For post processing this data into a quantitiy of interest, we first define a region of interest, a portion of the domain for which the solution is important. This region will be the same as our detector.

In [ ]:
roi_vol = RPPLogicalVolume(infx=True,
                           infy=True,
                           zmin=7.75, zmax=8.25)

### Computing a Quantity of Interest

Having defined a ROI, we now create a `FieldFunction`. In OpenSn we define a `FieldFunction` for the response we will like to calcualte. In this case we are looking for:

$$
\text{QoI} = \int_\mathcal{D} \, \int_{(4\pi)}  \, \sigma^g_{det}(\vec{r}) \psi^g(\vec{r},\vec\Omega) \, d^3r \, d\Omega = \int_\mathcal{D} \, \sigma^g_{det}(\vec{r}) \phi^g(\vec{r}) \, d^3r
$$

Thus, in OpenSn we will generate a scalar field function using `GetScalarFieldFunctionList` with a `sum` over the ROI. 


In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=True)
ffi = FieldFunctionInterpolationVolume()
ffi.SetOperationType("sum")
ffi.SetLogicalVolume(roi_vol)
ffi.AddFieldFunction(fflist[0])
ffi.Initialize()
ffi.Execute()

fwd_qoi = ffi.GetValue()

With our field function defined, we can also export the multi-group scalar flux, $\phi^g$, to a .vtu file using ``ExportMultipleToVTK``.

In [ ]:
FFGrid = FieldFunctionGridBased
FFGrid.ExportMultipleToVTK([fflist[0]], "Flux/FwdFlux_p")

## Adjoint Solver
### Adjoint Source

For solving the 1D transport problem via the Adjoint operator we first define our adjoint source $q^{\dagger}$ to be the detector volume, which we re-define as the `roi_vol` and apply a source strength of 1.0. (By default, adjoint sources are isotoropically distributed in angle across the souce domain)

In [ ]:
adj_src = VolumetricSource(logical_volume=roi_vol, group_strength=[1.0])

### Adjoint Options 
For executing the adjoint solver we reassign our solver options including,
+ A toggle for the adjoint 
+ Updating the volumetric sources

Note: Once the adjoint data has been save locally you do not need to run `Execute` to run the response evaluator.

In [ ]:
phys.SetOptions(
    adjoint = True,
    spatial_discretization = "pwld",
    volumetric_sources = [adj_src],
)
ss_solver.Execute()

### Evaluate Response

When evaluating response functions via the adjoint its efficiency derives from having access to the adjoint solution $\phi^{\dagger,g}$. For this, we export the adjoint flux moments to a .h5 file using `WriteFluxMoments`. The adjoint data is then saved to a location of our choosing.  

In [ ]:
data_dir = path+"/Data"
if not os.path.isdir(data_dir):
    os.makedirs(data_dir)
    
phys.WriteFluxMoments(data_dir+"/AdjFlux_p")

With the adjoint data in hand, we now create a `ResponseEvaluator`. 

We now supply two additional options to the `evaluator`, the `buffers` (adjoint data) and the `sources`. For computing the detector response we make use of `EvaluateResponse` which given a buffer name will compute an inner product of the adjoint flux $\phi^{\dagger,}$ against the forward source.    

In [ ]:
fwd_src = {'material': [{'block_id': 1, 'strength': [1.0]}]}

evaluator = ResponseEvaluator(lbs_problem = phys)
evaluator.SetOptions(
    buffers = [{
        'name': 'detector', 
        'file_prefixes': {'flux_moments': data_dir+'/AdjFlux_p'}
    }],
    sources = fwd_src
)

adj_resp = evaluator.EvaluateResponse("detector")

We export the adjoint flux solution to a .vtu using `ExportMultiToVTK`, the same as previously done for the "forward" flux. 

In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=False)
FFGrid = FieldFunctionGridBased
FFGrid.ExportMultipleToVTK([fflist[0][0]], "Flux/AdjFlux_p")

## Solution

The resulting flux solution for the forward and adjoint problem is shown below: 

![Flux](images/VolSrc.png)

For the QoI both methods gave the same solution:

QoI = 0.1463856233322801

## Finalize (for Jupyter Notebook only)

In Python script mode, PyOpenSn automatically handles environment termination. However, this
automatic finalization does not occur when running in a Jupyter notebook, so explicit finalization
of the environment at the end of the notebook is required. Do not call the finalization in Python
script mode, or in console mode.

Note that PyOpenSn's finalization must be called before MPI's finalization.


In [ ]:
from IPython import get_ipython

def finalize_env():
    Finalize()
    MPI.Finalize()

ipython_instance = get_ipython()
if ipython_instance is not None:
    ipython_instance.events.register("post_execute", finalize_env)